# ETL Pipelines and Data Transformation

ETL (Extract, Transform, Load) is a fundamental concept in data engineering. This notebook explores advanced ETL patterns and data transformation techniques.

## Advanced Data Extraction

Data can come from various sources:
- APIs (REST, GraphQL)
- Databases (SQL, NoSQL)
- Files (CSV, JSON, Parquet)
- Streams (Kafka, message queues)

In [ ]:
import pandas as pd
import json
from datetime import datetime, timedelta
import random

# Simulating multiple data sources

# Source 1: Sales data from CSV
sales_data = {
    'order_id': [1001, 1002, 1003, 1004, 1005],
    'customer_id': [101, 102, 103, 101, 104],
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Laptop'],
    'quantity': [1, 2, 1, 1, 2],
    'price': [1200.00, 25.00, 75.00, 300.00, 1200.00],
    'order_date': ['2025-01-15', '2025-01-16', '2025-01-16', '2025-01-17', '2025-01-18']
}

# Source 2: Customer data from JSON
customer_data = [
    {'customer_id': 101, 'name': 'John Doe', 'email': 'john@example.com', 'country': 'USA'},
    {'customer_id': 102, 'name': 'Jane Smith', 'email': 'jane@example.com', 'country': 'UK'},
    {'customer_id': 103, 'name': 'Bob Johnson', 'email': 'bob@example.com', 'country': 'Canada'},
    {'customer_id': 104, 'name': 'Alice Brown', 'email': 'alice@example.com', 'country': 'USA'}
]

print("Data sources loaded")

## Extract Phase

Create extraction functions for different sources:

In [ ]:
class DataExtractor:
    """Extract data from various sources"""
    
    @staticmethod
    def extract_sales(data):
        """Extract sales data"""
        df = pd.DataFrame(data)
        print(f"Extracted {len(df)} sales records")
        return df
    
    @staticmethod
    def extract_customers(data):
        """Extract customer data"""
        df = pd.DataFrame(data)
        print(f"Extracted {len(df)} customer records")
        return df

# Extract data
extractor = DataExtractor()
sales_df = extractor.extract_sales(sales_data)
customers_df = extractor.extract_customers(customer_data)

print("\nSales Data:")
print(sales_df.head())
print("\nCustomer Data:")
print(customers_df.head())

## Transform Phase

Transformations include:
- Data cleaning
- Type conversions
- Aggregations
- Joins and merges
- Enrichment

In [ ]:
class DataTransformer:
    """Transform and clean data"""
    
    @staticmethod
    def clean_sales(df):
        """Clean and transform sales data"""
        # Convert date column to datetime
        df['order_date'] = pd.to_datetime(df['order_date'])
        
        # Calculate total amount
        df['total_amount'] = df['quantity'] * df['price']
        
        # Add processing timestamp
        df['processed_at'] = datetime.now()
        
        # Extract date components
        df['year'] = df['order_date'].dt.year
        df['month'] = df['order_date'].dt.month
        df['day'] = df['order_date'].dt.day
        
        return df
    
    @staticmethod
    def clean_customers(df):
        """Clean and transform customer data"""
        # Standardize email to lowercase
        df['email'] = df['email'].str.lower()
        
        # Standardize country codes
        df['country'] = df['country'].str.upper()
        
        # Add customer segment based on ID
        df['segment'] = df['customer_id'].apply(
            lambda x: 'Premium' if x % 2 == 0 else 'Standard'
        )
        
        return df
    
    @staticmethod
    def join_data(sales_df, customers_df):
        """Join sales and customer data"""
        merged = sales_df.merge(
            customers_df, 
            on='customer_id', 
            how='left'
        )
        return merged

# Transform data
transformer = DataTransformer()
sales_cleaned = transformer.clean_sales(sales_df)
customers_cleaned = transformer.clean_customers(customers_df)

print("Cleaned Sales Data:")
print(sales_cleaned.head())

In [ ]:
# Join the datasets
final_data = transformer.join_data(sales_cleaned, customers_cleaned)
print("\nMerged Data:")
print(final_data)

## Data Aggregation

Create summary statistics and aggregations:

In [ ]:
# Aggregate by customer
customer_summary = final_data.groupby('customer_id').agg({
    'order_id': 'count',
    'total_amount': 'sum',
    'quantity': 'sum'
}).reset_index()

customer_summary.columns = ['customer_id', 'order_count', 'total_spent', 'items_purchased']

print("Customer Summary:")
print(customer_summary)

In [ ]:
# Aggregate by product
product_summary = final_data.groupby('product').agg({
    'order_id': 'count',
    'quantity': 'sum',
    'total_amount': 'sum'
}).reset_index()

product_summary.columns = ['product', 'order_count', 'units_sold', 'revenue']
product_summary = product_summary.sort_values('revenue', ascending=False)

print("\nProduct Summary:")
print(product_summary)

## Load Phase

Load transformed data to destination:

In [ ]:
class DataLoader:
    """Load data to various destinations"""
    
    @staticmethod
    def load_to_csv(df, filename):
        """Load data to CSV file"""
        df.to_csv(filename, index=False)
        print(f"Data loaded to {filename}")
    
    @staticmethod
    def load_to_json(df, filename):
        """Load data to JSON file"""
        df.to_json(filename, orient='records', indent=2)
        print(f"Data loaded to {filename}")
    
    @staticmethod
    def load_to_parquet(df, filename):
        """Load data to Parquet file"""
        df.to_parquet(filename, index=False)
        print(f"Data loaded to {filename}")

# Load data
loader = DataLoader()
loader.load_to_csv(final_data, 'sales_enriched.csv')
loader.load_to_json(customer_summary, 'customer_summary.json')

## Complete ETL Pipeline

Putting it all together:

In [ ]:
class ETLPipeline:
    """Complete ETL Pipeline"""
    
    def __init__(self):
        self.extractor = DataExtractor()
        self.transformer = DataTransformer()
        self.loader = DataLoader()
    
    def run(self, sales_source, customer_source, output_prefix='etl_output'):
        """Run the complete ETL pipeline"""
        print("=" * 50)
        print("Starting ETL Pipeline")
        print("=" * 50)
        
        try:
            # Extract
            print("\n1. EXTRACT PHASE")
            sales_df = self.extractor.extract_sales(sales_source)
            customers_df = self.extractor.extract_customers(customer_source)
            
            # Transform
            print("\n2. TRANSFORM PHASE")
            sales_cleaned = self.transformer.clean_sales(sales_df)
            customers_cleaned = self.transformer.clean_customers(customers_df)
            merged_data = self.transformer.join_data(sales_cleaned, customers_cleaned)
            print(f"Transformed and merged {len(merged_data)} records")
            
            # Load
            print("\n3. LOAD PHASE")
            self.loader.load_to_csv(merged_data, f'{output_prefix}_final.csv')
            
            print("\n" + "=" * 50)
            print("✓ ETL Pipeline completed successfully!")
            print("=" * 50)
            
            return merged_data
            
        except Exception as e:
            print(f"\n✗ ETL Pipeline failed: {str(e)}")
            raise

# Run the pipeline
pipeline = ETLPipeline()
result = pipeline.run(sales_data, customer_data, 'sales_pipeline')

## Data Quality Checks

Always validate data quality:

In [ ]:
class DataQualityChecker:
    """Validate data quality"""
    
    @staticmethod
    def check_nulls(df):
        """Check for null values"""
        nulls = df.isnull().sum()
        if nulls.sum() > 0:
            print("⚠ Null values found:")
            print(nulls[nulls > 0])
        else:
            print("✓ No null values found")
    
    @staticmethod
    def check_duplicates(df, key_columns):
        """Check for duplicate records"""
        duplicates = df.duplicated(subset=key_columns).sum()
        if duplicates > 0:
            print(f"⚠ {duplicates} duplicate records found")
        else:
            print("✓ No duplicates found")
    
    @staticmethod
    def check_data_types(df):
        """Check data types"""
        print("\nData Types:")
        print(df.dtypes)
    
    @staticmethod
    def generate_report(df, name="Dataset"):
        """Generate quality report"""
        print(f"\n{'=' * 50}")
        print(f"Data Quality Report: {name}")
        print(f"{'=' * 50}")
        print(f"Total Records: {len(df)}")
        print(f"Total Columns: {len(df.columns)}")
        DataQualityChecker.check_nulls(df)
        DataQualityChecker.check_data_types(df)

# Run quality checks
checker = DataQualityChecker()
checker.generate_report(result, "Sales Pipeline Output")

## Incremental Processing

Process only new or changed data:

In [ ]:
def incremental_load(new_data, existing_data, key_column):
    """Load only new records"""
    new_df = pd.DataFrame(new_data)
    existing_df = pd.DataFrame(existing_data)
    
    # Find new records
    new_records = new_df[~new_df[key_column].isin(existing_df[key_column])]
    
    print(f"Found {len(new_records)} new records")
    
    # Combine
    combined = pd.concat([existing_df, new_records], ignore_index=True)
    
    return combined

# Example
new_sales = {
    'order_id': [1006, 1007],
    'customer_id': [102, 103],
    'product': ['Keyboard', 'Mouse'],
    'quantity': [1, 3],
    'price': [75.00, 25.00],
    'order_date': ['2025-01-19', '2025-01-19']
}

updated = incremental_load(new_sales, sales_data, 'order_id')
print(f"\nTotal records after incremental load: {len(updated)}")

## Exercises

1. Add data validation rules (e.g., price > 0, valid email)
2. Implement error handling for each ETL phase
3. Create a pipeline that handles multiple file formats
4. Add logging to track pipeline execution
5. Implement a rollback mechanism for failed loads

## Best Practices for ETL

1. **Idempotency**: Same input should always produce same output
2. **Incremental Processing**: Process only changed data
3. **Data Validation**: Validate at each stage
4. **Error Handling**: Gracefully handle and log errors
5. **Performance**: Optimize for large datasets
6. **Monitoring**: Track metrics and failures
7. **Documentation**: Document transformations and business logic